In [1]:
import math

In [52]:
class Value():

    def __init__(self, data, _children=(), _op="", label=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.grad = 0
        self._backward = lambda: None

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        
        return out

    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "-")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        
        return out

    def __rmul__(self, other):
        return self * other

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += 1 - t**2
        out._backward = _backward
        return out 

    def backward(self):
        
        rev_topo = []
        visited = set()
        def build_rev_topo(v):
            rev_topo.append(v)
            for child in v._prev:
                build_rev_topo(child)
        build_rev_topo(self)
        
        self.grad = 1.0
        for node in rev_topo:
            node._backward()

    def exp(self):
        x = self.data
        t = math.exp(x)
        out = Value(t, (self, ), 'exp')

        def _backward():
            self.grad += t * out.grad
        out._backward = _backward

        return out

    def __pow__(self, other):
        out = Value(self.data**other, (self, ), 'pow')

        def _backward():
            
        out._backward = _backward
        
        return out

SyntaxError: incomplete input (1689662446.py, line 76)

In [50]:
x1, x2 = Value(2.0, label='x1'), Value(0.0, label='x2')
w1, w2 = Value(-3.0, label='w1'), Value(1.0, label='w2')
x1w1 = x1 * w1; x1w1.label = 'x1*w1'
x2w2 = x2 * w2; x2w2.label = 'x2w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
b = Value(6.881373587, label='b')
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

In [51]:
a = Value(3.0, label='a')
a.exp()

Value(data=20.085536923187668)